# Solar Filament Segmentation 2026 — end-to-end solution

YOLO11-seg proposals + U-Net semantic refinement, with mutually exclusive masks.

## 1. Setup

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, cv2, torch
BASE = Path('/data/research/sota_method_2026-10-24/solar_filament')
sys.path.insert(0, str(BASE/'code'))

## 2. Load the two models

Weights are stored in `runs/` inside this repository; no training is needed.

In [ ]:
from ultralytics import YOLO
import segmentation_models_pytorch as smp
yolo = YOLO(str(BASE/'runs/s1280/weights/best.pt'))
ck = torch.load(BASE/'runs/unet1024/best.pt', map_location='cpu', weights_only=False)
unet = smp.Unet(encoder_name='resnet34', encoder_weights=None, in_channels=3, classes=1)
unet.load_state_dict(ck['model']); unet.eval().cuda()

## 3. Predict the test split and write the submission

The refinement step re-thresholds each YOLO proposal against the U-Net probability field inside a dilated ROI; masks are then made mutually exclusive, which the platform requires.

In [ ]:
import subprocess, sys
BASE = '/data/research/sota_method_2026-10-24/solar_filament'
subprocess.run([sys.executable, f'{BASE}/code/predict_refine.py',
                '--yolo', f'{BASE}/runs/s1280/weights/best.pt',
                '--conf', '0.5', '--uthr', '0.85',
                '--out', f'{BASE}/out/submission.csv'], check=True)

## 4. Validate the submission contract

In [ ]:
subprocess.run([sys.executable, f'{BASE}/code/check_submission.py',
                f'{BASE}/out/submission.csv'], check=True)